In [23]:
import os
import pyarrow.parquet as pq


path = 'C:/Users/megaz/PycharmProjects/Lab-Studia-Semestr-3/data/yellow_tripdata_2025-01.parquet'
if not os.path.exists(path):
    raise FileNotFoundError(path)

try:
    # prefer disabling pandas metadata to avoid extension registration errors
    table = pq.read_table(path, use_pandas_metadata=False)
except TypeError:
    # older pyarrow may not accept the arg
    table = pq.read_table(path)

df = table.to_pandas()

In [24]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# USUWANIE BRAKÓW DANYCH
df = df.dropna()

# FEATURE ENGINEERING
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

df["speed"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

# USUWANIE OUTLIERÓW
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < df["fare_amount"].quantile(0.99))]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < df["trip_distance"].quantile(0.99))]
df = df[(df["trip_duration_min"] > 0) & (df["trip_duration_min"] < df["trip_duration_min"].quantile(0.99))]

target = "fare_amount"

X = df.drop(columns=[target])
y = df[target]

# STANDARYZACJA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.select_dtypes(include="number"))

# PODZIAŁ TRAIN / TEST
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

# PIPELINE
numeric_features = ["trip_distance", "trip_duration_min", "speed"]
categorical_features = ["VendorID", "payment_type", "pickup_hour"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features)
    ]
)

pipeline = Pipeline([
    ("preprocess", preprocessor)
])

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# =========================
# FEATURE ENGINEERING
# =========================

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

df["speed"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

# usunięcie oczywistych błędów
df = df[df["trip_duration_min"] > 0]
df = df[df["trip_distance"] > 0]
df = df[df["fare_amount"] > 0]

# =========================
# USUWANIE OUTLIERÓW
# =========================

for col in ["fare_amount", "trip_distance", "trip_duration_min"]:
    upper = df[col].quantile(0.99)
    df = df[df[col] < upper]

# =========================
# TARGET
# =========================

target = "fare_amount"

X = df.drop(columns=[target, "tpep_pickup_datetime", "tpep_dropoff_datetime"])
y = df[target]

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================
# FEATURES
# =========================

numeric_features = [
    "trip_distance",
    "trip_duration_min",
    "speed"
]

categorical_features = [
    "VendorID",
    "payment_type",
    "pickup_hour"
]

# =========================
# NUMERIC PIPELINE
# =========================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# =========================
# CATEGORICAL PIPELINE
# =========================

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# =========================
# COLUMN TRANSFORMER
# =========================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# =========================
# MODEL + PIPELINE
# =========================

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

# =========================
# TRAINING
# =========================

pipeline.fit(X_train, y_train)

# =========================
# PREDICTIONS
# =========================

y_pred = pipeline.predict(X_test)

# =========================
# RMSE
# =========================

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)